# Basic Seamless example

This notebook ports the basic workflow, filesystem mounts, Jupyter widgets and HTTP shares from the legacy `basic-example.ipynb` to modern Seamless.

Run all cells with the **seamless1** Python kernel. The widget section needs the optional `seamless-workflow[jupyter]` dependencies; HTTP shares need `seamless-workflow[share]`. Files are created in a temporary directory and cleaned up at the end. You can run this notebook without an interactive frontend; viewing and dragging the widgets requires JupyterLab or another widget frontend.

## 1. Build a workflow

A Context holds cells and transformers. Connect inputs through `pins` and connect the transformer's `result` to an output cell. `compute()` waits for the graph to finish.

In [ ]:
from seamless import Cell
from seamless_workflow import Context

ctx = Context()

def add(a, b):
    return a + b

ctx.a = Cell(celltype="plain")
ctx.a.set(10)
ctx.b = Cell(celltype="plain")
ctx.b.set(20)
ctx.add = add
ctx.add.pins.a = ctx.a
ctx.add.pins.b = ctx.b
ctx.c = Cell(celltype="plain")
ctx.c = ctx.add.result
ctx.compute(timeout=30)
assert ctx.c.value == 30
print("Initial sum:", ctx.c.value)

In [ ]:
ctx.a = ctx.a.value + 5
ctx.compute(timeout=30)
assert ctx.c.value == 35
print("Updated sum:", ctx.c.value)

## 2. Mount cells to files

The input cells use bidirectional mounts (`rw`). The computed output uses a write-only mount (`w`), which is legal on a cell with an incoming connection. `authority="cell"` lets the existing cell value win the initial attachment decision.

`ctx.mounts.sync()` is the attachment barrier. Call it before computing after an external edit, and after computing before reading an output file.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="seamless-basic-")
files = Path(workspace.name)
a_path, b_path, c_path = (files / name for name in ("a.json", "b.json", "c.json"))
ctx.a.mount(a_path, mode="rw", authority="cell")
ctx.b.mount(b_path, mode="rw", authority="cell")
ctx.c.mount(c_path, mode="w")
ctx.mounts.sync(timeout=10)
assert json.loads(a_path.read_text()) == 15
assert json.loads(b_path.read_text()) == 20
assert json.loads(c_path.read_text()) == 35
print("Mounted files:", files)

In [ ]:
# An editor outside Python can make the same change.
a_path.write_text("7\n")
ctx.mounts.sync(timeout=10)
assert ctx.a.value == 7
ctx.compute(timeout=30)
ctx.mounts.sync(timeout=10)
assert ctx.c.value == 27
assert json.loads(c_path.read_text()) == 27
print("Sum after file edit:", ctx.c.value)

### Edit transformer code through a mounted cell

Attachments require a whole cell node. To edit transformer code, mount a Python cell and connect that cell to the transformer's code handle.

In [ ]:
code_path = files / "code.py"
original_code = "def add(a, b):\n    return a + b\n"
code_path.write_text(original_code)
ctx.code = Cell(celltype="python")
ctx.code.mount(code_path, mode="r")
ctx.add.code = ctx.code
ctx.compute(timeout=30)
ctx.mounts.sync(timeout=10)
assert ctx.c.value == 27

code_path.write_text("def add(a, b):\n    return a + b + 1\n")
ctx.mounts.sync(timeout=10)
ctx.compute(timeout=30)
ctx.mounts.sync(timeout=10)
assert ctx.c.value == 28
assert json.loads(c_path.read_text()) == 28
print("Sum after code edit:", ctx.c.value)

# Restore addition before the widget demonstration.
code_path.write_text(original_code)
ctx.mounts.sync(timeout=10)
ctx.compute(timeout=30)
ctx.mounts.sync(timeout=10)
assert ctx.c.value == 27

## 3. Control cells from Jupyter

Use the free functions `traitlet(cell)` and `output(cell)`. A hub can hold several widget links. `link()` is bidirectional; `connect()` only displays the cell value.

Each cell supports one file slot and one widget hub slot. Keep the mounts active: slider edits update the input files, and computed values reach both the output file and the output widget. Each session spaces cell deliveries by at least two-thirds of a second and coalesces pending changes to the latest value; the synchronization barrier waits for delivery.

Move either slider and click **Update sum** to synchronize the inputs and compute the result. The output replaces its content on each complete delivery. Widget updates and hub observers run on transport worker threads.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from seamless_workflow.jupyter import traitlet, output

a = widgets.IntSlider(min=-10, max=30, description="a")
b = widgets.IntSlider(min=-10, max=30, description="b")
a_text = widgets.IntText(description="a (display)")
traitlet(ctx.a).link(a)
traitlet(ctx.a).connect(a_text)
traitlet(ctx.b).link(b)
c = output(ctx.c)

refresh = widgets.Button(description="Update sum")
def update_sum(button=None):
    ctx.mounts.sync(timeout=10)
    ctx.compute(timeout=30)
    ctx.mounts.sync(timeout=10)
refresh.on_click(update_sum)

display(a, a_text, b, refresh, c)
update_sum()
assert a.value == a_text.value == 7
assert b.value == 20
assert c.output_instance.outputs

In [ ]:
# Exercise both directions while the kernel is in blocking Context calls.
a.value = 7
b.value = 8
update_sum()
assert ctx.a.value == 7
assert ctx.b.value == 8
assert ctx.c.value == 15
assert a_text.value == 7
assert c.output_instance.outputs
assert "15" in str(c.output_instance.outputs)

ctx.a = 3
ctx.mounts.sync(timeout=10)
assert a.value == a_text.value == 3
update_sum()
assert ctx.c.value == 11
assert json.loads(a_path.read_text()) == 3
assert json.loads(b_path.read_text()) == 8
assert json.loads(c_path.read_text()) == 11
print("Widget edits also reached the mounted files")

## 4. Share cells over HTTP

Share the same `a`, `b`, and `c` cells already connected to widgets and file mounts. The browser page uses `a` and `b` as writable inputs and displays read-only `c`; graph reactivity computes the sum without a browser-side `compute()` call. The page is also a read-only `text` cell, mounted from `index.html` and shared as a top-level URL.

The link below opens the page on the local share server. Move a slider in either the browser or this notebook to see both views update.

In [ ]:
import seamless
from pathlib import Path
from html import escape
from IPython.display import HTML, display
import seamless_workflow
from seamless_workflow import shareserver

if shareserver.url is None:
    shareserver.configure(host="127.0.0.1", port=0)
workflow_source = Path(seamless_workflow.__file__).resolve().parent.parent
share_page_path = workflow_source / "examples/share-sliders/index.html"
if not share_page_path.is_file():
    raise FileNotFoundError(f"Slider page is missing: {share_page_path}")
ctx.page = Cell(celltype="text")
ctx.page.mount(share_page_path, mode="r")
ctx.a.share(readonly=False)
ctx.b.share(readonly=False)
ctx.c.share()
ctx.page.share(path="index.html", toplevel=True)
share_page_url = ctx.page.share.url
display(HTML(f'<a href="{escape(share_page_url, quote=True)}" target="_blank" rel="noreferrer">Open the shared sliders</a>'))

## 5. Clean up

For interactive exploration, leave the following cell unexecuted until you finish using the sliders. Closing the Context detaches its widgets, file mounts and shares. `seamless.close()` then stops the process-wide share server.

In [ ]:
refresh.on_click(update_sum, remove=True)
ctx.close()
seamless.close()
workspace.cleanup()
print("Context closed and temporary files removed")